# Dunnhumby seed 43: 선택 M5 내부 N/V 점수 기여 진단

**같은 M5 선택 체크포인트(150 epoch)**를 두 번 평가합니다. 하나는 원래 점수(ID+N+V), 하나는 추론 시 N/V 점수항만 가린 ID 성분 점수입니다. 신규상품 개발구간의 후보·정답 Top-10 이동과 전체·저/중/고CLV 지표를 기록합니다. 학습, 가중치 변경, 다른 epoch 선택, final test, holdout은 없습니다.

ID 성분 점수는 **M1이 아니며 새 추천모형도 아닙니다**. 이 비교는 선택된 M5 안에서 N/V의 직접적인 점수 기여를 설명할 뿐, M4와 M5의 차이를 N/V에 인과적으로 귀속하지 않습니다. 반복 노출 개발분할의 단일 시드이므로 유의성·일반화도 주장하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_m5_linear_nv_original_m4_lambda025_seed43_v1/reports/result.json'
if not REPORT.is_file():
    if os.path.ismount('/content/drive'):
        raise RuntimeError('Drive는 연결됐지만 정확한 결과를 찾을 수 없습니다. 계정과 REPORT 경로를 확인하세요.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 평가와 학습은 시작하지 않았습니다.') from exc
    if not REPORT.is_file():
        raise RuntimeError('Drive 연결 후에도 정확한 결과를 찾을 수 없습니다. REPORT 경로를 확인하세요.')
SOURCE_COMMIT = '5b707fcf69de6748e21a4de65ff5c16dcb1e3b1b'
REPO = Path('/content/clv-m5-nv-score-audit-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'clv_m4_m5_top10_movement_diagnostic', 'clv_m5_linear_nv_original_m4_lambda025_screen', 'clv_history_linear_nv_model'):
    if name in sys.modules:
        assert Path(sys.modules[name].__file__).resolve().parent == REPO.resolve(), '다른 소스가 로드됐습니다. 런타임을 재시작하세요: ' + name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_m5_nv_score_ablation_diagnostic as diagnostic
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_m5_nv_score_component_seed43_v1'
print('진단 버전:', diagnostic.VERSION, '| 재학습 없음')

In [ ]:
report, cfg = diagnostic.movement._verify_report(REPORT)
arm = diagnostic.movement._one_arm(report, diagnostic.movement.screen.MODEL_ID)
checkpoint = Path(arm['checkpoint'])
if not checkpoint.is_file() or diagnostic.movement.file_sha256(checkpoint) != arm['checkpoint_sha256']:
    raise RuntimeError('선택 M5 체크포인트가 없거나 바뀌었습니다. 재학습하지 않고 중단합니다.')
print('출처 JSON SHA256:', diagnostic.movement.SOURCE_RESULT_SHA256)
print('선택 M5:', arm['model_id'], '| epoch', arm['selected_epoch'], '| checkpoint', checkpoint)
print('ID-only는 같은 M5의 진단 점수이며 M1이 아닙니다.')

In [ ]:
paths = diagnostic.run(REPORT, OUT)
absolute = pd.read_csv(paths['absolute'])
comparison = pd.read_csv(paths['comparison'])
summary = pd.read_csv(paths['summary'])
print('1) 전체·CLV별 모든 지표:', paths['absolute'])
display(absolute)
print('2) N/V 직접 점수항 유무 차이:', paths['comparison'])
display(comparison)
print('3) Top-10 정답 이동:', paths['summary'])
display(summary)
print(json.dumps(paths, ensure_ascii=False, indent=2))
print('이 결과는 M2의 훈련효과·M4 대비 차이의 인과귀속이나 새 모델 성능 판정이 아닙니다.')

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = OUT/'m5_nv_score_component_seed43.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as zipped:
    for path in paths.values():
        zipped.write(path, arcname=Path(path).name)
print('진단 ZIP:', archive)
files.download(str(archive))